# Práctica para evaluación módulo 'Large Language Models'
## RAG simple (sin multi-turno)

### Carga de API_KEY desde archivo de claves
Se cargan las API keys proporcionadas por el profesor (`GOOGLE_API_KEY` y `OPENAI_API_KEY`) de archivo .env en la raíz del proyect 

In [8]:
# Carga de variables de entorno (carga variables de /.env: GOOGLE_API_KEY y OPENAI_API_KEY)
import os
from load_dotenv import load_dotenv
load_dotenv()

True

### Elección de modelos
Dado que se plantea un RAG, se necesita un modelo para generación y otro para la generación de los embeddings que contendrán en contexto derivado de la documentación aportada
Dado que no se requieren grandes capaciades y que se está en un entorno con un capacidad de uso de tokens limitada, se escogen:
- `GENERATION_MODEL`:
    **gemini-2.5-flash-lite**: *Our most cost-efficient multimodal model, offering the fastest performance for high-frequency, lightweight tasks. Gemini 2.5 Flash-Lite is best for high-volume classification, simple data extraction, and extremely low-latency applications where budget and speed are the primary constraints.*
- `EMBEDDING_MODEL`m :
    **gemini-embedding-001**: *A specialized engine for high-dimensional vector representation, providing efficient numerical mapping of text and images. The Gemini Embedding model is best for semantic search, document retrieval, and recommendation systems that require fast, scalable similarity calculations across large datasets.*    

In [9]:
# Elección modelos
# gemini-2.5-flash-lite:
# Our most cost-efficient multimodal model, offering the fastest performance for high-frequency, lightweight tasks. Gemini 2.5 Flash-Lite is best for high-volume classification, simple data extraction, and extremely low-latency applications where budget and speed are the primary constraints.
GENERATION_MODEL = 'gemini-2.5-flash-lite'


EMBEDDING_MODEL = 'gemini-embedding-001'
# gemini-embedding-001:
# A specialized engine for high-dimensional vector representation, providing efficient numerical mapping of text and images. The Gemini Embedding model is best for semantic search, document retrieval, and recommendation systems that require fast, scalable similarity calculations across large datasets.

### Instanciar modelos
Se escoge como `LangChain` como entorno en vistas a usar LangGraph más adelante, por la propiedad que tiene de estructurar un chat (sisntema multi-turno)

En `GENERATION_MODEL` se establecen los siguientes parámetros:
- `temperature`=**0.2**. Se elige un valor bajo porqué se prima la fiabilidad del contenido a la forma. Esto es una respuesta quzá menos elaborada en términos de lenguage natural pero con mayor fiabilidad en que el contenido se ajuste a lo esperado. El motivo es que en una fase inicial del proyecto y con un llm ligero se quiere poder descartar al máximo las posibles fuentes de error. No se especifian ni top-P ni top-K porqué son hasta cierto punto redundantes con la temperatura y así no generar posibles inconsistencias en lo que el moelo entienda
- `max_output_tokens`=**150**. Se escoge un número bajo para no correr el riesgo de vaciar la cuota antes de finalizar el ejercicio. En todo caso, cuando se tenga todo funcionando, ya se aumentará para las útlimas pruebas
- `response_mime_type`: **'text/plain'**. El por defecto, texto plano. En estr punto se pretende un chat con el usuario, no redirigir a ninguna herramienta

In [10]:
# Instanciar los modelos
from langchain.chat_models import init_chat_model
from langchain_google_genai import GoogleGenerativeAIEmbeddings

llm = init_chat_model(GENERATION_MODEL,
                      model_provider="google_genai",
                      temperature=0.2,         # baja temperatura, se incentivan respuestas deterministas
                      #top_p=0.3,               # solamente toma en cuentra la sigueinte palabara más probable: puede resultar en repetición palabaras pero se asgura al ausencia de fallos
                      max_output_tokens=150,    # por no quedarse sin crédito en la práctica
                      response_mime_type='text/plain')
embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)

### Cargar documentación con el contexto
De momento se carga el pdf, tal cual, con `PyPDFLoader`. El formato de texto absolutamente plano que se obtiene no permite hacer un chunking que aproveche la estructura que proveen los títulos. Se ve que el modelo no funciona muy bien con esto, aun probando distintos tamaños y overlap. Se deja para una fase posterior cargar el pdf previamiente convertido a markdown para solucionarlo

In [11]:
# Cargar el documento
from langchain_community.document_loaders import PyPDFLoader

pages = PyPDFLoader('data/TENERIFE.pdf').load()

/tmp/ipykernel_8888/2625867445.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [12]:
pages[0].metadata

{'producer': 'PyPDF',
 'creator': 'Microsoft Word',
 'creationdate': '2025-07-13T20:00:01+00:00',
 'title': 'Microsoft Word - TENERIFE.docx',
 'moddate': '2025-07-13T20:00:01+00:00',
 'source': 'data/TENERIFE.pdf',
 'total_pages': 25,
 'page': 0,
 'page_label': '1'}

### Chunking
Se usa `RecursiveCharacterTextSplitter` que usa la puntuación para ajustar donde corta el chunk
En el documento, se ha visto que, más o menos, el trozo mas largo de contenido simánticamente homogéneo es de unos 900 carácteres. También los hay de mucho más cortos. Se han probado varias opciones en longitud y overlap, sin conseguir que ninguna sea capaz de hacer un retrive correcto a 'cuales son las playas de la Orotava'

Para poder tracear mejor el resultado se añade en metadata un identificador del chunk y su tamaño. El nombredel documento y la página deonde se ha extraído el chunk ya los pone por defecto PyPDFLoader en 'source' y 'page_label'

In [13]:
# Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80,
    add_start_index=True,
)

chunks = text_splitter.split_documents(pages)

In [14]:
for i, chunk in enumerate(chunks):
    chunk.metadata['chunk']=i
    chunk.metadata['len']=len(chunk.page_content)

chunks[0].metadata

{'producer': 'PyPDF',
 'creator': 'Microsoft Word',
 'creationdate': '2025-07-13T20:00:01+00:00',
 'title': 'Microsoft Word - TENERIFE.docx',
 'moddate': '2025-07-13T20:00:01+00:00',
 'source': 'data/TENERIFE.pdf',
 'total_pages': 25,
 'page': 0,
 'page_label': '1',
 'start_index': 0,
 'chunk': 0,
 'len': 324}

### Creación de la base de datos vectorial
De momento se usa `InMemoryVectorStore`, esto es, no se crea una base de datos persistente
Se intentó con `FAISS`, pero daba errores y no se consiguió qe funcionase


In [15]:
# Creación embedings, que se guardan en base de datos vectorial
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)
document_ids = vector_store.add_documents(chunks)

In [16]:
# Probar el retrieval de la vector_store
question = "cuales son las playas de la Orotava"
chunks = vector_store.similarity_search(question, k=3)

for chunk in chunks:
    print(f"chunk number: {chunk.metadata['chunk']}")
    print(f"souce: {chunk.metadata['source']}")
    print(f"source page: {chunk.metadata['page_label']}")
    print(chunk.page_content)
    print(80*'=')

chunk number: 16
souce: data/TENERIFE.pdf
source page: 9
o En las fiestas de La Orotava (consideradas Fiestas de Interés Turístico 
Nacional) se hace una alfombra gigante de arenas y tierras procedentes del 
Teide y tiene el récord Guinness de mayor tapiz de tierra del mundo. 
A La Orotava pertenecen tres playas de las que os aconsejo visitar al menos una: 
 
o La Playa del Bollullo
chunk number: 11
souce: data/TENERIFE.pdf
source page: 8
o Playa de Benijo [vídeo - ubicación] 
Playa de arena negra, si el día está despejado, de los atardeceres más 
espectaculares que podéis ver en la isla. 
Para llegar a esta playa, aparcar por aquí y, para acceder a la playa, se accede 
desde aquí (puede ser que os encontréis con el acceso cortado con una valla, 
pero da igual, podéis acceder igualmente). 
• La Orotava
chunk number: 13
souce: data/TENERIFE.pdf
source page: 9
De La Orotava os recomendaría aparcar en esta explanada e ir caminando hacia la 
conocida como Plaza de Anita (no es su nombre of

In [17]:
'''
# Embeddings
from langchain_google_vertexai import VertexAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)

try:
    vectorstore = FAISS.load_local(
        'data', 
        embeddings, 
        allow_dangerous_deserialization=True)  # Required if loading a previously saved local index
except:
    vectorstore = FAISS.from_documents( pages, embeddings)
    vectorstore.save_local('data/faiss_index')
'''

"\n# Embeddings\nfrom langchain_google_vertexai import VertexAIEmbeddings\nfrom langchain_community.vectorstores import FAISS\n\nembeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)\n\ntry:\n    vectorstore = FAISS.load_local(\n        'data', \n        embeddings, \n        allow_dangerous_deserialization=True)  # Required if loading a previously saved local index\nexcept:\n    vectorstore = FAISS.from_documents( pages, embeddings)\n    vectorstore.save_local('data/faiss_index')\n"

### Creación del prompt template con LangChain
Con ello se puede encapsular de una vez las instrucciones de sistema y poder entrar como variables pregunta y contexto obtenido en la fase de 'retrieval'

En el mensaje de 'system':
- se especifica que únicamente se tenga en cuenta el contexto proporcionado por la documentación incluida en el RAG. No se quieren 'interferencias' externas, se quiere evaluar la capacidad de manejar la documentación dada
- se pide que si en el contexto no encuentra la información, que claramente lo diga, que no invente
- se pide que cite la fuente (se va a incluir un campo 'Fuente' en los metadatos de los chunk

In [18]:
# Prompt template
from langchain_core.prompts import ChatPromptTemplate

make_prompt = ChatPromptTemplate.from_messages(
    [
        (
             'system', 
             'Eres un asistente para dar información sobre lugares de interés en Tenerife'
             'Responde únicamente usando este contexto:\n{context}\n'
             'Si el contexto no contiene la respuesta, di claramente: información no disponible en el contexto'
             'No inventes datos'
             'Cita Fuentes al final',
        ),
        (
            'user',
            'Pregunta:\n{question}',
        )
    ]
)  

### Generación del string de contexto a partir de los chunk obtenidos en 'retrieval'
El contexto debe pasarse como un string al `prompt template`. Se incluyen los metadatos que interessan (para cada chunk, documento fuente, página del documento e identificador de chunk) para poder auditar la respuesta

In [19]:
# Generación de string de contexto a partir de chunks devueltos por similarity_search
def make_context_string(chunks):
    context = ""
    for chunk in chunks:
        context = context + f"Fuente: {chunk.metadata['source']},\
        Página: {chunk.metadata['page_label']},\
        chunk: {chunk.metadata['chunk']}\
        \n{chunk.page_content}\n\n"
    return context
        

### Constucción de la función de query
Tiene como parámetros el númoero de chunks a usar para contexto, la prenguta de usuario y, para poder auditar la respuesta, la posibilidad de que además de la respuesta de también el contexto usado
Ejectuta primero el retrieval y después la llamada al modelo generativo con el contexto obtenido
Retorna un diccionario con:
- `question`: la pregunta de usuario
- `answer`: la respuesta dada por la query
- `context`: el contexto que usa el modelo generativo
- `prompt`: el prompt pasado al modelo, que incluye las instrucciones de sistema, la pregunta y el contexto

In [20]:
def TenerifePOI(question: str, k: int = 3):
    
    chunks = vector_store.similarity_search(question, k=k)
    context = make_context_string(chunks)

    prompt = make_prompt.invoke({'question': question, 'context': context})
    llm_output = llm.invoke(prompt)

    return {
        "question": question,
        "answer": llm_output.content,
        "context": context,
        "prompt": prompt,
    }

### Prueba de la query
Falla en la pregunta 'playas en la Orotava'. Son Bollullo, Patos y Ancona. Se piensa que esto se podría mejorar si el chunking tomase los títulos: la información tiene un sentido, de título en adelante, no hacia los dos lados; Las 3 playas están entre dos títulos, el conteniendo el primero 'Orotava'

Se comprueba que funciona correctamente el devolver la fuente y responder que no dispone de la información cuando se pregunta por cosas fuera de la documentación del RAG

In [22]:
res = TenerifePOI('playas en la Orotava',
                  k=3)
print(res['answer'])

En La Orotava puedes visitar la Playa del Bollullo.

Fuente: data/TENERIFE.pdf, Página: 9


In [23]:
res = TenerifePOI('playas en Cambrils',
                  k=3)
print(res['answer'])

Información no disponible en el contexto.
